# XAI Enhancer Module - Colab Runner

This notebook allows you to run the XAI Enhancer module on Google Colab. It handles environment setup, sample data creation (using Hugging Face Datasets), and model downloading.

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision pillow numpy pandas tqdm grad-cam datasets huggingface_hub timm

# Confirm we are in the correct directory
import os
print(f"Current working directory: {os.getcwd()}")
if os.path.basename(os.getcwd()) != "XAI_Enhancer_module":
    print("⚠️ Warning: Please make sure you are in the 'XAI_Enhancer_module' directory.")
    if os.path.exists("XAI_Enhancer_module"):
        os.chdir("XAI_Enhancer_module")
        print(f"Changed directory to: {os.getcwd()}")

## 2. Create Sample Dataset (5000 Images)

We will stream the first 5000 images from the **ImageNet-1k validation set** using Hugging Face Datasets.

**Important**: ImageNet-1k is a gated dataset. You need to:
1.  Have a Hugging Face account.
2.  Accept terms at [https://huggingface.co/datasets/ILSVRC/imagenet-1k](https://huggingface.co/datasets/ILSVRC/imagenet-1k).
3.  Enter your Access Token below.

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
from pathlib import Path
from tqdm import tqdm
import json
import urllib.request
import os

# --- 1. Login to Hugging Face ---
# This will prompt for your token if not already logged in
# Check for environment variable or manual token
hf_token = os.environ.get("HF_TOKEN")
manual_token = "" # @param {type:"string"}

if hf_token:
    print("Found HF_TOKEN in environment, logging in...")
    login(token=hf_token, add_to_git_credential=False)
elif manual_token:
    print("Using manual token...")
    login(token=manual_token, add_to_git_credential=False)
else:
    print("Please enter your Hugging Face Access Token when prompted (or if a widget appears):")
    login(add_to_git_credential=False)

def create_imagenet_sample_from_hf(target_count=5000, base_path="imagenet_val_sample"):
    print(f"\nStarting download of {target_count} images from ImageNet-1k validation set...")
    
    # Load streaming dataset so we don't finish disk space
    # Using ILSVRC/imagenet-1k as requested
    try:
        print("Loading dataset from ILSVRC/imagenet-1k...")
        ds = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True, trust_remote_code=True)
    except Exception as e:
        print(f"\n❌ Error loading dataset: {e}")
        print("did you accept the manual terms at https://huggingface.co/datasets/ILSVRC/imagenet-1k ?")
        return None

    # --- Synset Mapping Handling ---
    # The evaluation scripts rely on LOC_synset_mapping.txt being present in the parent directory.
    mapping_path = None
    possible_paths = [
        "../LOC_synset_mapping.txt",
        "./LOC_synset_mapping.txt",
        "LOC_synset_mapping.txt"
    ]
    
    for p in possible_paths:
        if Path(p).exists():
            mapping_path = Path(p)
            print(f"Found synset mapping at: {mapping_path}")
            break

    # Download the official class index JSON for our own lookup
    # Try multiple sources in case one is down
    urls = [
        "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
        "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json",
        "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
    ]
    
    class_index = None
    for url in urls:
        try:
            print(f"Attempting to download class index from: {url}")
            with urllib.request.urlopen(url) as response:
                class_index = json.load(response)
            print("✅ Successfully downloaded class index.")
            break
        except Exception as e:
            print(f"⚠️ Failed to download from {url}: {e}")
            
    if class_index is None:
        print("❌ Could not download class index from any source.")
        return None

    # If text file missing, re-create it from the JSON so imports work
    if not mapping_path:
        mapping_path = Path("../LOC_synset_mapping.txt")
        print(f"⚠️ Synset mapping file missing. Generating it at {mapping_path}...")
        try:
            with open(mapping_path, 'w') as f:
                for idx in range(1000):
                    # class_index indices are strings "0", "1"...
                    entry = class_index[str(idx)]
                    synset = entry[0]
                    name = entry[1]
                    # Format: nXXXXXX class_name
                    f.write(f"{synset} {name}\n")
            print("✅ Created synset mapping file.")
        except Exception as e:
            print(f"Failed to write mapping file: {e}")
            print(f"Created synset mapping file at {mapping_path}")
    else:
         print(f"Synset mapping already exists at {mapping_path}")

    # The evaluation scripts check for imagenet_val_sample but you can override
    base_dir = Path(base_path)
    base_dir.mkdir(parents=True, exist_ok=True)
    
    count = 0
    print("Streaming and saving images...")
    
    for sample in tqdm(ds, total=target_count):
        if count >= target_count:
            break
        
        img = sample['image']
        label_idx = sample['label'] # Integer 0-999
        
        # Get synset ID
        synset_id = class_index[str(label_idx)][0]
        
        # Create folder
        synset_dir = base_dir / synset_id
        synset_dir.mkdir(exist_ok=True)
        
        # Save image
        if img.mode != 'RGB':
            img = img.convert('RGB')
            
        save_path = synset_dir / f"val_{count}.JPEG"
        if not save_path.exists():
            img.save(save_path)
        
        count += 1
        
    print(f"\n✅ Successfully saved {count} images to {base_path}")
    return str(base_dir)

# Run the creation
dataset_path = create_imagenet_sample_from_hf(target_count=5000)
if dataset_path:
    print(f"Dataset ready at: {dataset_path}")
else:
    print("Failed to create dataset.")

## 3. Download Models

We need to create a local directory for models so the script can find them.

In [ ]:
from download_models import download_all_models

# Set a local cache directory
MODEL_CACHE_DIR = "./pytorch_models"

# Download models (this might take a few minutes)
download_all_models(custom_folder=MODEL_CACHE_DIR)

## 4. Run Evaluation (Batch Mode)

This section now runs the evaluation in batches of 300 images. It will:
1.  Process the Enhanced CAM method in chunks.
2.  Process the Standard CAM methods in chunks.
3.  Save intermediate JSON results for each batch.
4.  (Optional) Send an email notification after each batch.
5.  Aggregate all results at the end.

In [ ]:
if dataset_path:
    # --- BATCH EVALUATION WITH EMAIL NOTIFICATIONS ---
    import subprocess
    import sys
    import getpass
    import shlex
    import os
    
    # Configuration
    TOTAL_IMAGES = 5000 # Total images to evaluate
    BATCH_SIZE = 300    # Images per processing chunk (for restartability)
    EVAL_BATCH_SIZE = 512 # Batch size for GPU inference (Speed optimization)
    STEP_SIZE = 224      # Pixel step size (Precision vs Speed: 50 is precise, 224 is fast)
    MODEL_NAME = "resnet50"
    
    # metrics configuration
    STANDARD_METHODS = "HiResCAM" # Add others as needed
    
    print("\n📧 Email Notification Setup (Optional)")
    print("Leave blank to skip email notifications.")
    email_to = input("Recipient Email: ").strip()
    email_sender = "fnsaikia@gmail.com"
    email_password = ""
    
    if email_to:
        email_sender = input("Sender Email (Gmail): ").strip()
        email_password = getpass.getpass("Sender Password (App Password): ").strip()
    
    num_workers = str(os.cpu_count() or 4)
    print(f"\n🚀 Starting Batch Evaluation: {TOTAL_IMAGES} images in chunks of {BATCH_SIZE}")
    print(f"Model: {MODEL_NAME}")
    print(f"Processing Batch Size (GPU): {EVAL_BATCH_SIZE}")
    print(f"Evaluation Step Size: {STEP_SIZE}")
    print(f"Data Loader Workers: {num_workers}")
    
    # Helper to run command and stream output
    def run_command(cmd_list):
        process = subprocess.Popen(
            cmd_list,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
            universal_newlines=True
        )
        while True:
            output = process.stdout.readline()
            if output == '' and process.poll() is not None:
                break
            if output:
                print(output.strip())
        rc = process.poll()
        return rc
    
    # 1. Run Enhanced CAM in batches
    print(f"\n{'='*20} PHASE 1: ENHANCED CAM {'='*20}")
    for start_idx in range(0, TOTAL_IMAGES, BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, TOTAL_IMAGES)
        print(f"\n▶️ Processing Chunk: {start_idx} to {end_idx}")
        
        cmd = [
            "python", "imagenet_evaluation.py",
            "--model", MODEL_NAME,
            "--imagenet-path", dataset_path,
            "--eval-type", "enhanced-only",
            "--enhanced-cam-method", "HiResCAMEnhanced",
            "--model-cache-dir", "./pytorch_models",
            "--device", "cuda",
            "--layer-mode", "all",
            "--start-index", str(start_idx),
            "--end-index", str(end_idx),
            "--batch-size", str(EVAL_BATCH_SIZE),
            "--step-size", str(STEP_SIZE),
            "--save-intermediate",
            "--num-workers", num_workers
        ]
        
        if email_to:
            cmd.extend([
                "--email-to", email_to,
                "--email-sender", email_sender,
                "--email-password", email_password
            ])
            
        run_command(cmd)
        
    # 2. Run Standard Methods in batches
    print(f"\n{'='*20} PHASE 2: STANDARD METHODS {'='*20}")
    for start_idx in range(0, TOTAL_IMAGES, BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, TOTAL_IMAGES)
        print(f"\n▶️ Processing Chunk: {start_idx} to {end_idx}")
        
        cmd = [
            "python", "imagenet_evaluation.py",
            "--model", MODEL_NAME,
            "--imagenet-path", dataset_path,
            "--eval-type", "standard-only",
            "--methods"] + STANDARD_METHODS.split() + [
            "--model-cache-dir", "./pytorch_models",
            "--device", "cuda",
            "--start-index", str(start_idx),
            "--end-index", str(end_idx),
            "--batch-size", str(EVAL_BATCH_SIZE),
            "--step-size", str(STEP_SIZE),
            "--save-intermediate",
            "--num-workers", num_workers
        ]
        
        if email_to:
            cmd.extend([
                "--email-to", email_to,
                "--email-sender", email_sender,
                "--email-password", email_password
            ])
            
        run_command(cmd)

    # 3. Aggregate Results
    print(f"\n{'='*20} PHASE 3: AGGREGATION {'='*20}")
    # The default output dir for analysis results is ./analysis_results
    # Results are stored in {output_dir}/{model_name}_imagenet/
    results_dir = f"./analysis_results/{MODEL_NAME}_imagenet"
    
    agg_cmd = [
        "python", "imagenet_evaluation.py",
        "--model", MODEL_NAME,
        "--imagenet-path", dataset_path, # Still required by arg parser
        "--aggregate-dir", results_dir,
        "--output-csv-dir", "./csv_exports"
    ]
    
    if email_to:
        agg_cmd.extend([
            "--email-to", email_to,
            "--email-sender", email_sender,
            "--email-password", email_password
        ])
        
    run_command(agg_cmd)
    
else:
    print("Skipping evaluation as dataset was not created.")

## 5. View Results

Results are saved in `csv_exports` and `analysis_results`.

In [ ]:
import pandas as pd
import glob

# Find the latest CSV result
csv_files = glob.glob("csv_exports/*/*aggregated.csv")
if csv_files:
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"Reading results from: {latest_csv}")
    df = pd.read_csv(latest_csv)
    display(df)
else:
    # Fallback to standard ones if aggregation didn't happen
    csv_files = glob.glob("csv_exports/*/*.csv")
    if csv_files:
         latest_csv = max(csv_files, key=os.path.getctime)
         print(f"Reading results from: {latest_csv}")
         df = pd.read_csv(latest_csv)
         display(df)
    else:
        print("No CSV results found yet.")